In [ ]:
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from script.config import configure_runtime

configure_runtime("3")

import torch
from IPython.display import Image, display

from script import config as cfg
from script.air_inference import AirForecaster
from script.data_loader import load_demo_sample
from script.plot_snapshots import display_snapshots, plot_demo_snapshots
from script.surface_inference import SurfaceForecaster

device = "cuda:0" if torch.cuda.is_available() else "cpu"
init_date = datetime(cfg.YEAR, cfg.MONTH, cfg.DAY)
print(f"{device} | {cfg.OUTPUT_ROOT}")


In [ ]:
from script.wd_scores import evaluate_wd_rmse
from script.warmup import warmup_models

input_list, time_str, air_truth, surface_truth = load_demo_sample(
    year=cfg.YEAR, month=cfg.MONTH, day=cfg.DAY, time_steps=cfg.TIME_STEPS
)

air_model = AirForecaster(device=device)
surface_model = SurfaceForecaster(device=device, ngcm=air_model.ngcm)
warmup_models(air_model, surface_model, input_list, time_str)

air_rmse, surface_rmse, wrf_rmse = evaluate_wd_rmse()


In [ ]:
from script.plot_scores import plot_wd_rmse, png_paths

saved = plot_wd_rmse(air_rmse, surface_rmse, wrf_rmse, lead_hours=cfg.WRF_FORECAST_HOURS, save_dir=cfg.OUTPUT_ROOT / "figures_wb")
for path in png_paths(saved):
    display(Image(filename=str(path)))


In [ ]:
air_pred = air_model.predict(input_list, [time_str])
surface_pred = surface_model.predict(input_list, [time_str])

In [ ]:
cfg.save_tensor(air_pred, "air_pred.pt")
cfg.save_tensor(air_truth, "air_truth.pt")
cfg.save_tensor(surface_pred, "surface_pred.pt")
cfg.save_tensor(surface_truth, "surface_truth.pt")

In [ ]:
display_snapshots(plot_demo_snapshots(
    air_pred, air_truth, surface_pred, surface_truth, init_date=init_date,
))
